In [ ]:
# === ARRANQUE EN COLAB: árbol de carpetas de la sesión =====================
# Este cuaderno se escribió para correr desde la carpeta `notebook/` de su
# sesión, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese árbol, así que aquí se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/E04_segmentar_patrones"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesion EPE E4 - Segmentar clientes y encontrar patrones

**Curso "Herramientas de Ciencias de Datos" - Modalidad EPE - UPC - Facultad de Negocios**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonatanfigueroagil-creator/Herramientas-de-Ciencias-de-Datos/blob/master/Sesiones_EPE/E04_segmentar_patrones/notebook/EPE_S4_no_supervisado.ipynb)

> Enfoque EPE: se prioriza la **intuicion y la decision de negocio** sobre el
> formalismo matematico. Esta sesion cubre clustering + deteccion de anomalias
> y reglas de asociacion / market basket. La segmentacion se hace con
> **k-means como intuicion**; el agrupamiento jerarquico, DBSCAN y los detectores
> Isolation Forest / LOF se **mencionan como opciones**, sin su profundidad tecnica.

## Objetivos de aprendizaje
Al terminar la sesion, el participante es capaz de:
1. **Segmentar** clientes agrupando por similitud (analisis **RFM** + **k-means**) y **perfilar** cada segmento.
2. **Elegir** cuantos segmentos usar con el **coeficiente de silueta**, sin decidir por inspeccion visual.
3. **Detectar lo inusual** (clientes/transacciones anomalas) a nivel de intuicion (fraude, errores de registro).
4. **Descubrir productos que se compran juntos** con reglas de asociacion (**soporte, confianza, lift**) y proponer **cross-selling**.

## Mapa de la sesion
| # | Bloque | Datos |
|---|---|---|
| a | Segmentar clientes: agrupar por similitud (k-means) + RFM | Online Retail II - por cliente (140) |
| b | Cuantos segmentos elegir (silueta) y como perfilarlos | idem |
| c | Deteccion de anomalias: encontrar lo inusual | idem |
| d | Reglas de asociacion: canasta de mercado y cross-selling | Online Retail II - por factura (800 tickets) |
| e | Cierre: decision de negocio | -- |

**Materiales hermanos de esta sesion:** guia de laboratorio `laboratorio/GUIA_LABORATORIO_E04.docx`,
plantillas `plantillas/plantilla_rfm_segmentos.docx` y `plantillas/plantilla_top_reglas.docx`,
ejercicios `evaluacion/drills.docx`, entregable evaluable `evaluacion/entregable.docx` y
fuentes de actualidad las fuentes de actualidad de la sesión.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el núcleo científico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SÍ. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aquí solo se instala lo que Colab NO trae.
import sys

if "google.colab" in sys.modules:
    %pip install -q mlxtend

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los últimos decimales; el método y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "matplotlib": "3.11.1",
    "numpy": "2.5.1",
    "openpyxl": "3.1.5",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "seaborn": "0.13.2",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


In [ ]:
# Librerías de la sesión
import os, sys, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, mlxtend
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Estética (paleta de marca UPC: rojo + neutros)
UPC_RED, UPC_INK, UPC_GRAY = "#E4002B", "#2D2D2D", "#9AA0A6"
SEG_COLORS = ["#E4002B", "#2D2D2D", "#9AA0A6", "#F2A900"]  # 4 segmentos
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.titleweight": "bold",
                     "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("Librerías OK -", "Colab" if "google.colab" in sys.modules else "entorno local")
# Traza del stack: versiones EXACTAS con que se generaron los anclajes (reproducibilidad).
print(f"Stack: pandas={pd.__version__} numpy={np.__version__} matplotlib={plt.matplotlib.__version__} "
      f"seaborn={sns.__version__} sklearn={sklearn.__version__} mlxtend={mlxtend.__version__}")

In [ ]:
# === DATOS DE LA SESIÓN EMBEBIDOS (para Google Colab) ======================
# En Colab no existe la carpeta data/ del curso. Los CSV de la sesión viajan
# comprimidos DENTRO de este cuaderno y se escriben en el directorio de trabajo
# antes de cargarse: son los MISMOS bytes que en local (mismo SHA256), de modo
# que las cifras de la sesión no cambian. En local esta celda no hace nada.
import base64, gzip, os, sys

_EMBEBIDOS = {
    "online_retail_II_rfm_muestra.csv":  # 978,230 B  sha256=96b678af3bebf0d6...
        "...",
    "online_retail_baskets_muestra.csv":  # 951,194 B  sha256=a3745101de08f44a...
        "...",
}

def _escribir_embebidos(destinos=None):
    """Vuelca los CSV embebidos al directorio actual y a ../data: los cuadernos
    del curso leen de uno o de otro según como calculen su carpeta de datos."""
    if destinos is None:
        destinos = [os.getcwd(), os.path.join(os.getcwd(), "data"),
                    os.path.join(os.path.dirname(os.getcwd()), "data")]
    escritos = []
    for _dest in destinos:
        try:
            os.makedirs(_dest, exist_ok=True)
        except OSError:
            continue
        for _nombre, _b64 in _EMBEBIDOS.items():
            _ruta = os.path.join(_dest, _nombre)
            if not os.path.exists(_ruta):
                with open(_ruta, "wb") as _fh:
                    _fh.write(gzip.decompress(base64.b64decode(_b64)))
                escritos.append(_nombre)
    return sorted(set(escritos))

if "google.colab" in sys.modules:
    _e = _escribir_embebidos()
    print("Datos de la sesión listos en Colab:", ", ".join(_e) if _e else "ya estaban")


In [ ]:
# Rutas robustas (funcionan con nbconvert local y en Colab) y carga de datos VERIFICADA.
import urllib.request, hashlib

def _base_dir():
    if "google.colab" in sys.modules:
        return os.getcwd()
    d = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(d, "la ficha de la sesiónmd")):
            return d
        d = os.path.dirname(d)
    d = os.getcwd()
    return os.path.dirname(d) if os.path.basename(d).lower() == "notebook" else d

BASE = _base_dir()
DATA = os.getcwd() if "google.colab" in sys.modules else os.path.join(BASE, "data")
RES = os.path.join(BASE, "resultados"); FIG = os.path.join(BASE, "figuras")
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True); os.makedirs(DATA, exist_ok=True)
XLSX = os.path.join(RES, "E04_resultados.xlsx")

# --- Integridad de datos: SHA256 + esquema (misma guarda que data/descargar_datos.py) ---
# Los CSV son las MUESTRAS (<=1 MB) ya verificadas; su hash es identico. El cargador
# NO acepta un archivo cuyo SHA256 no coincida, de modo que la via primaria "Abrir en
# Colab" queda con la misma guarda de integridad que en local.
SHA256 = {
    "online_retail_II_rfm_muestra.csv": "96b678af3bebf0d6277e0e101c7fca0745b37c6d080a2158df0adc3bc58e0aef",
    "online_retail_baskets_muestra.csv": "a3745101de08f44a8fc67167e9075cb12427a776b53e7d0cdc503a78ddae2760",
}
ESQUEMA = {  # columnas minimas + nº de filas esperado por archivo
    "online_retail_II_rfm_muestra.csv":
        (["Invoice", "InvoiceDate", "Quantity", "Price", "Customer ID", "Country"], 18125),
    "online_retail_baskets_muestra.csv":
        (["Invoice", "StockCode", "Description", "Country"], 16979),
}
# Multi-mirror: rama master del repo del curso (carpeta E04) y mirrors adicionales
# que sirven las MISMAS muestras -> tolerancia a un 404 o renombre.
_RAW = ("https://raw.githubusercontent.com/jonatanfigueroagil-creator/"
        "Herramientas-de-Ciencias-de-Datos/master/")
MIRRORS = {
    "online_retail_II_rfm_muestra.csv": [
        _RAW + "Sesiones_EPE/E04_segmentar_patrones/data/online_retail_II_rfm_muestra.csv",
        _RAW + "Sesiones/S07_clustering_anomalias/data/online_retail_II_muestra.csv",
    ],
    "online_retail_baskets_muestra.csv": [
        _RAW + "Sesiones_EPE/E04_segmentar_patrones/data/online_retail_baskets_muestra.csv",
        _RAW + "Sesiones/S08_reglas_asociacion/data/online_retail_baskets_muestra.csv",
    ],
}

def _sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def _verificar_esquema(ruta, nombre):
    cols, nfilas = ESQUEMA[nombre]
    df = pd.read_csv(ruta)
    faltan = [c for c in cols if c not in df.columns]
    if faltan:
        raise ValueError(f"[{nombre}] faltan columnas {faltan}; hay {list(df.columns)}")
    if len(df) != nfilas:
        raise ValueError(f"[{nombre}] se esperaban {nfilas} filas y hay {len(df)}")

def asegurar(nombre):
    """Idempotente + integridad: descarga solo si falta o el hash no cuadra, prueba
    los mirrors en orden y exige SHA256 + esquema antes de aceptar el archivo."""
    ruta = os.path.join(DATA, nombre)
    esperado = SHA256[nombre]
    if os.path.exists(ruta) and _sha256(ruta) == esperado:
        return ruta                        # presente y verificado -> no re-descarga
    errores = []
    for url in MIRRORS[nombre]:
        try:
            raw = urllib.request.urlopen(url, timeout=60).read()
            tmp = ruta + ".part"
            with open(tmp, "wb") as f: f.write(raw)
            got = _sha256(tmp)
            if got != esperado:
                os.remove(tmp)
                raise ValueError(f"SHA256 no coincide ({got[:12]}... != {esperado[:12]}...)")
            os.replace(tmp, ruta)
            _verificar_esquema(ruta, nombre)   # ademas del hash, valida el esquema
            print(f"  + {nombre}: descargado y verificado (SHA256 + esquema).")
            return ruta
        except Exception as e:
            errores.append(f"{url.split('/master/')[-1]}: {e!r}")
    raise RuntimeError(
        f"No se pudo obtener {nombre} de ningun mirror. La muestra (<=1 MB) vive en "
        f"data/ del repo; alternativa: Online Retail II de UCI id 502 "
        f"(https://archive.ics.uci.edu/dataset/502). Detalle por mirror: {errores}")

rfm_src = pd.read_csv(asegurar("online_retail_II_rfm_muestra.csv"), parse_dates=["InvoiceDate"])
baskets = pd.read_csv(asegurar("online_retail_baskets_muestra.csv"))
print("Online Retail II - por cliente :", rfm_src.shape, "|", rfm_src["Customer ID"].nunique(), "clientes")
print("Online Retail II - por factura :", baskets.shape, "|", baskets["Invoice"].nunique(), "facturas")

---
## a) Segmentar clientes: agrupar por similitud (k-means) + RFM

**Caso de negocio.** Un e-commerce de regalos quiere **tratar distinto a clientes
distintos**: no tiene sentido enviar el mismo correo al mejor cliente y a uno que
compro una vez hace dos anos. **Segmentar** es agrupar a los clientes por
**similitud de comportamiento**, para disenar una accion por grupo.

**La intuicion de k-means (agrupar por similitud).** k-means es la forma mas
conocida de agrupar: se fija un numero de grupos **k**, se colocan **k centros** y
cada cliente se asigna al centro **mas cercano**; los centros se recolocan como el
"cliente promedio" de su grupo y se repite hasta que se estabilizan. No hay
etiquetas ni respuesta correcta previa: es **aprendizaje no supervisado**, se
descubre la estructura que ya esta en los datos. *(Opciones mas avanzadas -
agrupamiento **jerarquico** con dendrograma y **DBSCAN** por densidad - no se
desarrollan aqui; basta la intuicion de k-means.)*

**RFM: tres numeros que resumen a un cliente.** Para agrupar hace falta describir a
cada cliente. El modelo clasico **RFM** usa el historial de compras:
- **Recencia (R):** dias desde su **ultima compra** (menor = mas activo).
- **Frecuencia (F):** numero de **compras/pedidos**.
- **Monto (M):** **gasto total** del cliente.

Solo necesita el registro de transacciones - datos que toda empresa ya tiene.

**❓ Qué se quiere averiguar.** ¿Alcanza con tres números —cuándo compró por última vez, cuántas veces y cuánto gastó— para describir a un cliente, y se obtienen esos tres números de datos que la empresa ya tiene?

- **Qué decide:** si la segmentación es viable sin comprar información demográfica ni encuestar a nadie. Todo lo que sigue en la sesión se apoya sobre esta tabla.
- **Antes de mirar el resultado:** lo que importa aquí es la **forma** de las tres variables, no su promedio. Si Recencia, Frecuencia y Monto resultan **repartidos de forma pareja**, se agrupan tal cual. Si el Monto muestra **colas muy largas** —unos pocos clientes con casi todo el gasto—, la media deja de representar a nadie y hará falta transformar y estandarizar antes de medir distancias.

🔎 **Qué hace este código.** Reconstruye las tres variables **RFM** por cliente desde las transacciones crudas: agrupa por `Customer ID` y calcula **Recencia** (días desde la última compra respecto a una fecha de referencia), **Frecuencia** (número de facturas distintas) y **Monto** (suma de `Cantidad x Precio`). Convierte un historial de tickets en una tabla de clientes lista para segmentar.

In [ ]:
# Calcular R, F, M por cliente desde las transacciones (Online Retail II).
rfm_src["Amount"] = rfm_src["Quantity"] * rfm_src["Price"]
ref = rfm_src["InvoiceDate"].max() + pd.Timedelta(days=1)   # fecha de referencia
rfm = (rfm_src.groupby("Customer ID")
       .agg(Recency=("InvoiceDate", lambda s: (ref - s.max()).days),
            Frequency=("Invoice", "nunique"),
            Monetary=("Amount", "sum"))
       .reset_index())
print("Clientes:", len(rfm))
print(rfm[["Recency", "Frequency", "Monetary"]].describe().round(1).to_string())

📖 **Como leer esta salida (por que preparar los datos antes de agrupar).**
- **Colas largas.** El **Monto** va desde unas pocas libras hasta cientos de miles:
  unos pocos clientes concentran casi todo el gasto. La **media** enganaria; por eso
  al perfilar se usa la **mediana**, y antes de agrupar se aplica un `log` a F y M
  para comprimir esas colas.
- **Escalas distintas (imprescindible estandarizar).** Monto (miles), Frecuencia
  (unidades) y Recencia (dias) viven en escalas muy diferentes. k-means mide
  **distancias**: sin poner las tres variables en una **escala comun** (`StandardScaler`),
  el Monto dominaria y la segmentacion reflejaria las **unidades**, no el
  comportamiento. Es un error de preparacion silencioso y frecuente.

⚠️ **Alerta (escala / estandarizacion).** Sin estandarizar, los "segmentos" son
**rangos de gasto disfrazados**, no perfiles de comportamiento: la variable de mayor
magnitud (el Monto) domina la distancia. La unica excepcion es Isolation Forest,
que parte por cortes y no por distancias.

**❓ Qué se quiere averiguar.** ¿En cuántos grupos conviene partir la cartera de clientes? Nadie trae la respuesta correcta: no hay etiquetas contra las cuales validar.

- **Qué decide:** cuántas campañas distintas tendrá que diseñar y sostener marketing. Cada segmento de más es un mensaje más, un presupuesto más y un responsable más.
- **Antes de mirar el resultado:** si la silueta muestra un **pico claro** en algún k, la métrica decide sola. Si resulta **casi plana** —y aquí el rango entero entre k = 2 y k = 6 cabe en unas tres centésimas—, entonces la métrica **informa pero no decide**, y el número de segmentos se elige por lo que marketing pueda accionar. Las diferencias en la tercera cifra son ruido: cambiar solo la semilla ya las mueve.

🔎 **Qué hace este código.** Prepara los datos y **explora cuántos grupos** conviene. Aplica `log` a Frecuencia y Monto (comprime las colas largas), estandariza las tres variables a una **escala común** (`StandardScaler`) y ejecuta k-means para k de 2 a 6 anotando la **silueta** de cada uno. Aún no fija los segmentos: solo mide qué tan separada queda la agrupación con cada k.

In [ ]:
# Preparación: log en F y M (colas largas) + estandarización (escala común).
X = rfm[["Recency", "Frequency", "Monetary"]].copy()
X["Frequency"] = np.log1p(X["Frequency"])
X["Monetary"] = np.log1p(X["Monetary"])
Z = StandardScaler().fit_transform(X)

# Probar varios k y medir la calidad de la segmentación con la silueta.
#   random_state=42 : fija la semilla -> todos obtienen EXACTAMENTE los mismos segmentos.
#   n_init=10       : k-means arranca 10 veces con centros distintos y se queda con el
#                     mejor, para no depender de un arranque desafortunado.
sil = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Z)
    sil.append({"k": k, "silhouette": round(silhouette_score(Z, km.labels_), 4)})
sil = pd.DataFrame(sil)
print(sil.to_string(index=False))

# Incertidumbre por semilla: cambiando SOLO la semilla, k=4 se mueve un poco. Ese
# wiggle es mayor que la diferencia entre k contiguos (k=3 vs k=4) -> el "ranking" a
# la tercera cifra es ruido, no señal. (El rango ENTRE k=2..6 es aparte, ~0.03.)
sd = dict(zip(sil["k"], sil["silhouette"]))
sil_k4 = [round(silhouette_score(Z, KMeans(n_clusters=4, random_state=s, n_init=10).fit(Z).labels_), 4)
          for s in range(5)]
rango_semilla = round(max(sil_k4) - min(sil_k4), 4)
gap_k3_k4 = round(abs(sd[3] - sd[4]), 4)
print(f"\nSilueta k=4 con 5 semillas: {sil_k4} -> rango {rango_semilla}")
print(f"  ruido por semilla ({rango_semilla}) >= diferencia k=3 vs k=4 ({gap_k3_k4}) -> ese orden es ruido")

---
## b) Cuantos segmentos elegir (silueta) y como perfilarlos

**Coeficiente de silueta.** Mide, sin etiquetas, **que tan bien agrupado** esta cada
cliente: cerca de **+1** = bien dentro de su grupo y lejos de los demas; cerca de
**0** = en la frontera; **negativo** = probablemente mal asignado. Se prueba varios
**k** y se compara. Como referencia: ~0.5-0.7 estructura razonable; ~0.25-0.5
estructura debil (tipico de RFM, donde los grupos no estan perfectamente separados).

💡 **Intuicion.** La silueta equivale a verificar, para cada cliente, si se parece mas a su
grupo que al grupo vecino. Un promedio alto = grupos nitidos; uno bajo = fronteras
borrosas.

⚠️ **Alerta (falsa precision / la silueta informa, no decide).** No hay "segmentacion
verdadera" contra la cual validar. En esta ejecucion la silueta es **casi plana**: k=2≈0.389,
k=3≈0.401, k=4≈0.399, k=5≈0.416, k=6≈0.409 — todo el rango entre k cabe en **~0.03**.
Ademas, la diferencia entre **k contiguos es ruido**: k=3 (0.401) incluso **supera** al
k=4 elegido (0.399), y por apenas **~0.002**, menos que la variacion que produce cambiar **solo
la semilla** (~0.003, ver la comprobacion de 5 semillas de la celda anterior). Por eso las
diferencias a la tercera cifra **no** deben leerse como un "ascenso" real.
**Maximizar la silueta de forma mecanica** es enganoso: aqui el maximo esta en **k=5** (0.416) y el
minimo en **k=2** (0.389) — el ejemplo tipico de "maximizar de forma mecanica lleva a k=2" **no** aplica a
estos datos (k=2 es justamente el peor). Se elige **k = 4** **no** por la metrica —que es
plana— sino por ser **interpretable y accionable** para marketing.

**❓ Qué se quiere averiguar.** ¿Qué hay dentro de cada grupo y qué se hace con él? Un número de cluster no es un segmento: hace falta traducir cada grupo a un perfil con nombre y a una acción concreta.

- **Qué decide:** el plan de marketing del trimestre. De esta tabla se derivan cuatro decisiones distintas: a quién se fideliza, a quién se le vende más, a quién se acompaña hasta la segunda compra y a quién se reactiva o se descarta.
- **Antes de mirar el resultado:** si los cuatro perfiles resultan **parecidos entre sí**, la partición carece de utilidad por buena que sea la silueta. Si resultan **claramente distintos** en R, F y M, cada uno admite su propia acción. El perfil se lee con la **mediana** y no con la media: basta un cliente extremo para que el promedio de un segmento deje de describir a sus miembros.

🔎 **Qué hace este código.** Fija **k = 4**, asigna cada cliente a un segmento y **perfila** cada grupo con la **mediana** de R, F y M (robusta ante los gastos extremos). Luego nombra los segmentos por su perfil -*Campeones/VIP*, *Leales*, *Recientes/nuevos*, *Inactivos*- sin depender del número de etiqueta que k-means asigne.

In [ ]:
# Segmentar con k = 4 y PERFILAR cada grupo en unidades de negocio.
K = 4
rfm["segmento"] = KMeans(n_clusters=K, random_state=42, n_init=10).fit_predict(Z)

# Perfil por segmento con la MEDIANA (robusta ante colas), en unidades reales.
# Se reporta también la MEDIA (Monto_medio) SOLO para exponer cuánto la distorsionan
# los extremos: en el VIP la media (~63 700) triplica largo a la mediana (~13 600).
perfil = (rfm.groupby("segmento")
          .agg(n_clientes=("Customer ID", "size"),
               Recencia_mediana=("Recency", "median"),
               Frecuencia_mediana=("Frequency", "median"),
               Monto_mediano=("Monetary", "median"),
               Monto_medio=("Monetary", "mean"))
          .round(1))

# Nombrar los segmentos por su perfil (ranking robusto, no depende del nro de etiqueta):
#   VIP = mayor gasto medio ; Inactivos = mayor recencia (hace más que no compran) ;
#   de los dos restantes, Leales = mayor gasto ; el último = Recientes de bajo valor.
orden_gasto = perfil["Monto_medio"].sort_values(ascending=False).index.tolist()
vip = orden_gasto[0]
resto = [s for s in perfil.index if s != vip]
inactivos = perfil.loc[resto, "Recencia_mediana"].idxmax()
resto2 = [s for s in resto if s != inactivos]
leales = perfil.loc[resto2, "Monto_medio"].idxmax()
recientes = [s for s in resto2 if s != leales][0]
nombres = {vip: "Campeones / VIP", leales: "Leales de alto valor",
           recientes: "Recientes / nuevos de bajo valor", inactivos: "Inactivos / en riesgo"}
perfil["nombre"] = [nombres[s] for s in perfil.index]
rfm["nombre"] = rfm["segmento"].map(nombres)
print(perfil[["nombre", "n_clientes", "Recencia_mediana", "Frecuencia_mediana",
              "Monto_mediano", "Monto_medio"]].to_string())

# Cifra ajustada del VIP SIN la ballena: un solo cliente infla la MEDIA del segmento.
# Al quitarlo, la media de los otros VIP cae hasta casi igualar la MEDIANA -> por eso
# la cifra "típica" que se publica es la MEDIANA, no la media.
vip_df = rfm[rfm["segmento"] == vip]
ballena = vip_df["Monetary"].max()
media_sin_ballena = vip_df.loc[vip_df["Monetary"] < ballena, "Monetary"].mean()
print(f"\nVIP ({len(vip_df)} clientes): media=£{vip_df['Monetary'].mean():,.1f}  "
      f"mediana=£{vip_df['Monetary'].median():,.1f}  "
      f"media SIN la ballena (£{ballena:,.0f}, {ballena/vip_df['Monetary'].sum():.0%} del VIP)=£{media_sin_ballena:,.1f}")

📖 **Cómo leer esta salida (fichas de segmento y acción de marketing).** Cada segmento
se **nombra** por su perfil y se le asigna **una** acción (cifras de la ejecución):
- **Campeones / VIP** (6 clientes): compraron **hace pocos días**, con frecuencia muy alta y
  **gasto muy elevado**. Cifra típica = **mediana £13 620**; la **media es £63 727**, pero está
  **distorsionada** por un solo cliente (la *ballena*, £313 946 ≈ **82 %** del gasto VIP):
  al quitarlo, la media de los otros 5 VIP cae a **~£13 683**, casi igual a la mediana. Por
  eso se publica la **mediana**, no la media. Pocos, pero sostienen el negocio. Acción:
  **fidelizar** (trato preferente, acceso anticipado); **no** malgastar descuentos en quien
  ya compra.
- **Leales de alto valor** (~44): recientes, frecuentes y de **buen gasto**
  (mediana ~£2 400). Acción: **cross-selling / up-selling** y mantenerlos activos.
- **Recientes / nuevos de bajo valor** (~45): compraron **hace poco** pero **una o
  dos veces** y poco monto. Acción: **onboarding** e incentivo a la **segunda compra**.
- **Inactivos / en riesgo** (~45): **hace mucho** que no compran (recencia ~435 días),
  1 compra, bajo monto. Acción: **reactivación de bajo costo** o descarte si no responden.

💡 **Intuición (valor x tamaño).** El tamaño del segmento pondera la prioridad: una
campaña cara no se justifica para un grupo muy reducido, salvo que su **valor** lo amerite.
Los 6 VIP mueven ~70 % del gasto —pero **~58 % del total lo pone un único cliente**—, así
que "6 VIP = 70 %" no significa 6 clientes intercambiables: hay que **atender aparte a la
ballena** y no tratarla como cliente promedio.

**❓ Qué se quiere averiguar.** ¿Se sostiene la separación de los cuatro segmentos cuando se los mira en un plano, o solo existe dentro de la tabla?

- **Qué decide:** si la segmentación se puede presentar. Una figura donde cada grupo ocupa su zona convence a un comité; una nube sin estructura con cuatro colores superpuestos, no.
- **Antes de mirar el resultado:** en el mapa de recencia contra monto los **VIP** tendrían que caer arriba a la izquierda —compran hace poco y mucho— y los **inactivos** a la derecha. Si los colores aparecen **mezclados**, el corte en cuatro es más una construcción del algoritmo que un hecho del negocio. Y la curva de silueta al lado funciona como recordatorio: casi plana entre 4 y 6, de modo que la elección del número sigue siendo de negocio.

In [ ]:
# Figura 1: mapa de segmentos (Recencia vs Monto, tamaño = Frecuencia).
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for i, s in enumerate(perfil.index):
    d = rfm[rfm.segmento == s]
    ax.scatter(d["Recency"], d["Monetary"], s=20 + d["Frequency"] * 3,
               color=SEG_COLORS[i % len(SEG_COLORS)], alpha=0.7, edgecolor="white",
               linewidth=0.4, label=nombres[s])
ax.set_yscale("log")
ax.set_xlabel("Recencia (días desde la última compra)")
ax.set_ylabel("Monto total (GBP, escala log)")
ax.set_title("Segmentos de clientes (RFM + k-means) - tamaño = Frecuencia")
ax.legend(fontsize=8, title="Segmento")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "E04_segmentos_rfm.png"), bbox_inches="tight")
plt.show()

# Figura 2: silueta vs k (por qué se eligió k pequeño y accionable).
fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.plot(sil["k"], sil["silhouette"], "o-", color=UPC_RED, linewidth=2)
ax.axvline(K, color=UPC_GRAY, linestyle="--", linewidth=1)
ax.set_xlabel("Número de segmentos (k)"); ax.set_ylabel("Silueta media")
ax.set_title("Calidad de la segmentación según k (línea = k elegido)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "E04_silueta_k.png"), bbox_inches="tight")
plt.show()

📖 **Cómo leer esta salida.** La **primera figura** cruza recencia (eje X) y monto (eje Y, escala log), con el tamaño del punto = frecuencia: los *VIP* quedan arriba a la izquierda (compran hace poco y mucho) y los *inactivos* a la derecha. La **segunda figura** traza la silueta según k; la línea punteada marca el k elegido. La curva es casi plana entre 4 y 6, así que la métrica **no impone** el número: se eligen 4 segmentos por ser los más accionables.

---
## c) Deteccion de anomalias: encontrar lo inusual

**Idea (intuicion).** A veces lo valioso no es el grupo, sino el **caso raro**: una
transaccion de fraude, un error de registro (un precio de 0, una cantidad de 10 000),
o un cliente que no se parece a **ningun** otro. Detectar anomalias es marcar
**candidatos a revisar**, ordenando a cada observacion por **cuan inusual** es.

Un metodo habitual (**Isolation Forest**) da a cada cliente un **score de rareza**;
se revisa el porcentaje mas alto (p. ej. el 5 %). *Aqui se usa a nivel de intuicion
para "encontrar lo inusual"; su version tecnica -y alternativas como **LOF** o el
**ruido** de DBSCAN- no se desarrollan en esta sesion.* Una bandera de anomalia es una
**hipotesis para un humano**, no una sentencia de fraude.

🔎 **Que hace este codigo.** Ejecuta `IsolationForest` sobre el mismo RFM estandarizado,
asigna a cada cliente un **score de rareza** (mayor = mas inusual), marca como candidato
el ~5 % mas raro y lista los 6 clientes mas inusuales para revision. Dos argumentos
gobiernan el resultado: **`contamination=0.05`** = se asume que **~5 %** de los clientes
son anomalos, asi que se marca ese 5 % mas raro (es una **eleccion del analista** -cuanto
quiere revisar-, **no** un umbral estadistico); **`random_state=42`** fija la semilla del
bosque para que la lista de candidatos sea **reproducible** (misma ejecucion para todos).

In [ ]:
# Marcar el ~5% de clientes más inusuales sobre el mismo RFM estandarizado.
iso = IsolationForest(random_state=42, contamination=0.05).fit(Z)
rfm["score_rareza"] = -iso.score_samples(Z)          # mayor = más inusual
rfm["es_inusual"] = iso.predict(Z) == -1              # bandera binaria (candidato)
top_anom = (rfm.sort_values("score_rareza", ascending=False)
            .head(6)[["Customer ID", "nombre", "Recency", "Frequency", "Monetary", "score_rareza"]]
            .round(2))
print("Clientes más inusuales (candidatos a revisar):")
print(top_anom.to_string(index=False))
print("\nTotal marcados como inusuales:", int(rfm["es_inusual"].sum()), "de", len(rfm))

📖 **Como leer esta salida.** Los mas "inusuales" son de **dos tipos opuestos**, y ambos
merecen revision:
- El **cliente-ballena** (recencia ~9 dias, ~156 compras, ~314 000 de gasto): no es un
  error, es un cliente excepcional; conviene **atenderlo aparte** (y quiza excluirlo del
  modelo de segmentos para que no distorsione).
- Cuentas con **1 compra y monto minusculo** (~17 GBP de por vida): posibles **errores
  de registro** o clientes de una sola vez; utiles para **depurar** los datos antes de
  decidir.

⚠️ **Alerta (una anomalia distorsiona el promedio).** Un solo cliente de ~314 000 mueve
la media de un segmento entero: antes de sumarlo hay que **decidir si es genuino o un
error**. La bandera `es_inusual` es una **hipotesis a revisar**, no una sentencia; puede
acompanar al `segmento` como **senal** para modelos posteriores (E5).

---
## d) Reglas de asociacion: canasta de mercado y cross-selling

**Caso de negocio.** Que productos se **compran juntos**? Si al llevar A casi siempre
se lleva B, se pueden **colocar juntos**, diseñar una **oferta conjunta (*bundle*)** o recomendar "quienes
compraron A tambien compraron B" (**cross-selling**). El **analisis de canasta de
mercado** descubre esas co-ocurrencias en los tickets, sin necesitar datos
demograficos ni un modelo predictivo.

**Tres numeros para leer una regla `A -> B`:**
- **Soporte:** que fraccion de **todos** los tickets contienen A y B juntos. Mide
  **volumen** (que tan comun es el patron).
- **Confianza:** de los tickets que llevan **A**, que fraccion tambien lleva **B**. Es
  la **fiabilidad** de la sugerencia.
- **Lift (la metrica clave):** cuantas **veces mas** aparecen A y B juntos frente a lo
  esperado **por azar**. **lift = 1** independientes; **> 1** asociacion positiva
  (candidata a cross-selling); **< 1** sustitucion (comprar A **reduce** B). El lift
  **corrige el engano de la confianza**: una B muy popular da confianza alta aunque no
  haya asociacion real (regla **trivial**, lift ~ 1).

⚠️ **Alerta (la confianza engana -> decidir por lift).** Una **B muy popular** (una
bolsa que casi todos compran) da **confianza alta** sin asociacion real: es una **regla
trivial** con **lift ~ 1**. Por eso se **ordena por lift, nunca por confianza sola**, y
se descartan las triviales y las **espurias** (co-ocurrencias por azar, sin mecanismo de
negocio).

*(La preparacion `one-hot por ticket`, el algoritmo **Apriori** y su alternativa
**FP-Growth**, y medidas extra como **conviction**, no se detallan aqui; se usan
directamente con **mlxtend**.)*

🔎 **Que hace este codigo.** Construye la **canasta por factura** (productos distintos por
ticket), la convierte en una matriz **one-hot** (filas = facturas, columnas = productos),
ejecuta **Apriori** (`min_support = 0.02`) para hallar itemsets frecuentes y deriva las
**reglas de asociacion**, quedandose con el **top por lift** en formato legible.

In [ ]:
# 1) Armar la canasta por factura: lista de productos distintos por ticket.
canastas = baskets.groupby("Invoice")["Description"].apply(lambda s: sorted(set(s))).tolist()

# 2) One-hot booleano por ticket (filas = facturas, columnas = productos).
te = TransactionEncoder()
onehot = pd.DataFrame(te.fit_transform(canastas), columns=te.columns_)
print("Matriz one-hot (facturas x productos):", onehot.shape)

# 3) Itemsets frecuentes (Apriori) y reglas de asociación.
#    min_support=0.02 : solo patrones presentes en >=2 % de los tickets (~16 de 800).
#                       Es el PISO de volumen; más bajo llena de reglas anecdóticas.
itemsets = apriori(onehot, min_support=0.02, use_colnames=True)
reglas = association_rules(itemsets, metric="lift", min_threshold=1)
#    confidence>=0.3 : filtro de accionabilidad. Descarta sugerencias poco fiables
#                      (de cada 10 compras del antecedente, acierta en <3); deja las
#                      reglas "usables" para una recomendación. Baja el conteo a 38.
reglas = reglas[reglas["confidence"] >= 0.3].copy()
print("Itemsets frecuentes (soporte >= 0.02):", len(itemsets), "| reglas (confianza >= 0.3):", len(reglas))

# 4) Top reglas por lift, en formato legible.
def _txt(fs): return ", ".join(sorted(fs))
reglas["antecedente"] = reglas["antecedents"].apply(_txt)
reglas["consecuente"] = reglas["consequents"].apply(_txt)
top = (reglas.sort_values("lift", ascending=False)
       .head(8)[["antecedente", "consecuente", "support", "confidence", "lift"]]
       .rename(columns={"support": "soporte", "confidence": "confianza"})
       .round(3).reset_index(drop=True))
with pd.option_context("display.max_colwidth", 46):
    print(top.to_string(index=False))

In [ ]:
# Por qué el lift de la regla estrella es 32,73 (y no 45): reconciliar la fórmula.
# El "soporte" que se ve en la tabla (0,02) es el del CONJUNTO A y B juntos. El lift NO
# se calcula con ese, sino con el soporte del CONSECUENTE B solo (qué tan común es B en
# TODOS los tickets):  lift = confianza / soporte(consecuente B).
estrella = reglas.sort_values("lift", ascending=False).iloc[0]
N = onehot.shape[0]
sop_conj = estrella["support"]              # soporte del conjunto A y B
sop_cons = estrella["consequent support"]   # soporte del consecuente B (el que entra al lift)
conf = estrella["confidence"]
print(f"Regla estrella : {estrella['antecedente']} -> {estrella['consecuente']}")
print(f"  soporte(A y B juntos)  = {sop_conj:.4f}  (~{round(sop_conj*N)} de {N} tickets; en la tabla se ve 0,02)")
print(f"  soporte(consecuente B) = {sop_cons:.4f}  (~{round(sop_cons*N)} de {N} tickets)")
print(f"  confianza(A -> B)      = {conf:.3f}")
print(f"  lift = confianza / soporte(B) = {conf:.3f} / {sop_cons:.4f} = {conf/sop_cons:.2f}")
print(f"  (usar por error el soporte del CONJUNTO de la tabla: {conf:.2f} / 0,02 = {conf/0.02:.0f}, no {conf/sop_cons:.1f})")

📖 **Como leer esta salida (cross-selling).**
- Las reglas de mayor **lift** unen los **juegos de tazas "Regency Teacup and Saucer"**
  por color (rosa, verde, rosas): quien compra una taza tiene **~32 veces** mas
  probabilidad de llevar la de otro color que un cliente al azar, con **confianza ~0.8-0.9**.
  Es una oportunidad clara de cross-selling: **ofrecer el juego completo** o una **oferta conjunta (*bundle*)** de colores.
- **De donde proviene el 32,73** (celda anterior). El lift **no** usa el soporte del conjunto
  (0,02), sino el **soporte del consecuente**: la taza verde aparece en **0,0275** de los
  tickets (~22 de 800). Entonces `lift = confianza / soporte(consecuente) = 0,90 / 0,0275 =
  32,73`. Usar por error el soporte del conjunto (0,90 / 0,02) daria 45 y **no reconcilia**.
- **Cuidado con las reglas triviales.** Una regla puede tener confianza alta solo
  porque el producto es **muy popular** (una bolsa que casi todos compran); su **lift ~ 1**
  la identifica como no informativa. Por eso se **ordena por lift**, nunca por confianza sola.

⚠️ **Alerta (base delgada + asociacion != causalidad).** La regla estrella descansa en un
soporte de **0,02 = ~16 de 800 tickets** (el piso permitido): el lift 32,73 es **alto pero
impreciso** (poca muestra), asi que es una **candidata a validar**, no una certeza. Y el
lift dice que A y B **co-ocurren**, no que promover A **cause** comprar B (podrian compartir
una causa, como una campana estacional). La cifra de impacto es una **hipotesis a validar
con un experimento A/B** (E1); la causalidad formal se ve en E6.

In [ ]:
# --- Exportar TODOS los resultados a resultados/E04_resultados.xlsx ---
seg_out = perfil.reset_index()[["segmento", "nombre", "n_clientes", "Recencia_mediana",
                                "Frecuencia_mediana", "Monto_mediano", "Monto_medio"]]
with pd.ExcelWriter(XLSX, engine="openpyxl") as w:
    sil.to_excel(w, sheet_name="rfm_silhouette", index=False)
    seg_out.to_excel(w, sheet_name="rfm_segmentos", index=False)
    top_anom.to_excel(w, sheet_name="anomalias_top", index=False)
    top.to_excel(w, sheet_name="reglas_top", index=False)
print("Excel de resultados escrito en:", XLSX)
print("Hojas:", ["rfm_silhouette", "rfm_segmentos", "anomalias_top", "reglas_top"])

🔎 **Qué hace este código.** Genera las dos figuras de resultados **leyendo el Excel** (`E04_resultados.xlsx`), no los objetos en memoria -convención del curso para que la gráfica y la cifra publicada siempre coincidan-: el tamaño de cada segmento y el top de reglas por lift.

In [ ]:
# --- Figuras de RESULTADOS: se generan LEYENDO el Excel (convención del curso) ---
seg = pd.read_excel(XLSX, sheet_name="rfm_segmentos").sort_values("Monto_medio", ascending=False)
rg = pd.read_excel(XLSX, sheet_name="reglas_top")

# R1) Tamaño de cada segmento (nº de clientes).
fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.barh(seg["nombre"], seg["n_clientes"], color=SEG_COLORS[:len(seg)])
ax.bar_label(bars, padding=3)
ax.invert_yaxis(); ax.set_xlabel("N clientes"); ax.set_title("Tamaño de cada segmento de clientes")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "res_E04_segmentos_tamano.png"), bbox_inches="tight")
plt.show()

# R2) Top reglas de cross-selling por lift.
rg2 = rg.sort_values("lift").tail(6)
etiquetas = [f"{a[:22]} -> {c[:22]}" for a, c in zip(rg2["antecedente"], rg2["consecuente"])]
fig, ax = plt.subplots(figsize=(7.4, 4))
bars = ax.barh(etiquetas, rg2["lift"], color=UPC_RED)
ax.bar_label(bars, fmt="%.1f", padding=3)
ax.set_xlabel("Lift (veces sobre el azar)"); ax.set_title("Top reglas de cross-selling por lift")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "res_E04_top_reglas_lift.png"), bbox_inches="tight")
plt.show()
print("Figuras de resultados guardadas en:", FIG)

---
## e) Cierre: decisión de negocio

**Del dato a la decisión.** Con datos que la empresa **ya tiene** (solo transacciones),
la sesión produjo dos activos accionables:
1. **Segmentos de clientes (RFM + k-means):** cuatro grupos con **nombre y acción** -
   fidelizar a los **VIP**, hacer cross-sell a los **leales**, activar la segunda
   compra de los **nuevos** y reactivar a los **inactivos** - más una bandera de
   **clientes inusuales** a revisar.
2. **Reglas de cross-selling (market basket):** productos que se compran juntos
   (los **juegos de tazas**, lift ~32) para ofertas conjuntas (*bundles*), colocación y recomendaciones.

**Recomendación tipo.** Concentrar el presupuesto donde el **valor x tamaño** rinde:
retener a los **Campeones/VIP** y activar el **cross-selling** de los juegos de tazas
sobre los **Leales**; tratar las reglas de alto lift como **hipótesis** y confirmarlas
con un **A/B** antes de escalar (el lift es asociación, no causa).

### Entregable de la sesión
Ver `evaluacion/entregable.docx`: **segmentación RFM + fichas de segmento con acciones** y
**top-N reglas de cross-selling** con lectura de negocio, usando las plantillas
`plantillas/plantilla_rfm_segmentos.docx` y `plantillas/plantilla_top_reglas.docx`; rúbrica
**vigesimal (total = 20)**. Los ejercicios de práctica están en `evaluacion/drills.docx`.

### Drills (enunciados; se resuelven en `evaluacion/drills.docx`)
1. **Definir y perfilar tres segmentos** de clientes a partir de RFM.
2. **Marcar y revisar** las observaciones más anómalas.
3. **Seleccionar tres reglas** de asociación accionables y justificar su uso.

### Para seguir explorando (fuentes de actualidad)
Casos y estudios recientes sobre segmentacion, RFM y motores de recomendacion /
cross-selling, con enlaces verificados, en las fuentes de actualidad de la sesión.

### Bibliografia (lecturas de la sesion)
- James, Witten, Hastie & Tibshirani. *An Introduction to Statistical Learning* (ISLR), cap. 12.4 (agrupamiento). [statlearning.com](https://www.statlearning.com/)
- Provost, F. & Fawcett, T. (2013). *Data Science for Business*, cap. 6 (similitud, vecindarios y agrupamiento). O'Reilly.
- Fader, Hardie & Lee (2005). *RFM and CLV*. Journal of Marketing Research 42(4).
- Agrawal & Srikant (1994). *Fast Algorithms for Mining Association Rules* (Apriori). Proc. VLDB.

---
*Cuaderno de la Sesion EPE E4 - Herramientas de Ciencias de Datos - UPC. Datos: Online
Retail II (UCI id 502, CC BY 4.0). Todas las cifras
provienen de la ejecucion de este cuaderno y de `resultados/E04_resultados.xlsx`.*